In [1]:
import pandas as pd
import sqlite3
import pycountry

### LOAD DATABASE

In [2]:
conn = sqlite3.connect("../data/flights.db")

### CREATE QUERIES TO HAVE DATASETS

In [3]:
query = """
SELECT 
    MIN(flight_number) as flight_number,
    departure_airport,
    arrival_airport,
    departure_scheduled,
    MIN(departure_estimated) as departure_estimated,
    MIN(departure_delay_minutes) as departure_delay_minutes,
    arrival_scheduled,
    MIN(status) as status,
    date,
    MIN(created_at) as created_at,
    COUNT(*) as duplicate_count
FROM flights
WHERE status != 'unknown'
GROUP BY 
    departure_airport,
    arrival_airport,
    departure_scheduled,
    arrival_scheduled,
    date
ORDER BY departure_airport, arrival_airport, departure_scheduled
"""

flights_df = pd.read_sql_query(query, conn)

query = "SELECT * FROM airports"
airports_df = pd.read_sql_query(query, conn)

query = "SELECT * FROM train_routes"
routes_df = pd.read_sql_query(query, conn)



### FILTER FLIGHTS TO BE ONLY BETWEEN EUROPEAN COUNTRIES

In [4]:
# Keep only flights where both departure and arrival airports are in Europe (continent == 'EU')

# Get European airports from airports_df
_europe_airports = airports_df[airports_df["continent"] == "EU"].copy()

# Build the set of European IATA airport codes
_europe_iata_codes = set(_europe_airports["iata_code"].dropna().astype(str))

before_flights = len(flights_df)

# Filter flights_df so that both endpoints are European airports
flights_df = flights_df[
    flights_df["departure_airport"].isin(_europe_iata_codes)
    & flights_df["arrival_airport"].isin(_europe_iata_codes)
].copy()

after_flights = len(flights_df)

print(f"Filtered flights_df to only intra-European flights: {before_flights} -> {after_flights}")

Filtered flights_df to only intra-European flights: 118646 -> 91342


### EXTRA FILTERING FOR GERMANY AND LUXEMBOURG (HAD 150 STATIONS EACH)

In [5]:
### FILTER STOPS IN GERMANY AND LUXEMBOURG

import json
from collections import defaultdict

# Normalize station names to canonical forms (maps variants to representative names)
def normalize_station_name(station_name):
    """Normalize station name variants to canonical forms."""
    if not station_name:
        return station_name
    
    name_lower = station_name.lower()
    
    # Frankfurt - keep main station and airport only
    if 'frankfurt' in name_lower and ('main' in name_lower or 'hbf' in name_lower):
        return 'Frankfurt (Main) Hbf'
    if 'frankfurt' in name_lower and ('flughafen' in name_lower or 'airport' in name_lower):
        return 'Frankfurt Flughafen'
    
    # Berlin - keep main station and Südkreuz (major hub)
    if 'berlin' in name_lower and ('hauptbahnhof' in name_lower or 'hbf' in name_lower):
        return 'Berlin Hbf'
    if 'berlin' in name_lower and 'südkreuz' in name_lower:
        return 'Berlin Südkreuz'
    if 'flughafen ber' in name_lower or 'ber airport' in name_lower:
        return 'Flughafen BER'
    
    # Munich - keep main station and airport
    if ('münchen' in name_lower or 'munich' in name_lower) and ('hauptbahnhof' in name_lower or 'hbf' in name_lower):
        return 'München Hbf'
    if ('münchen' in name_lower or 'munich' in name_lower) and ('flughafen' in name_lower or 'airport' in name_lower):
        return 'München Flughafen'
    
    # Hamburg - keep main station
    if 'hamburg' in name_lower and ('hauptbahnhof' in name_lower or 'hbf' in name_lower):
        return 'Hamburg Hbf'
    
    # Cologne - keep main station
    if ('köln' in name_lower or 'cologne' in name_lower) and ('hauptbahnhof' in name_lower or 'hbf' in name_lower):
        return 'Köln Hbf'
    
    # Stuttgart - keep main station
    if 'stuttgart' in name_lower and ('hauptbahnhof' in name_lower or 'hbf' in name_lower):
        return 'Stuttgart Hbf'
    
    # Düsseldorf - keep main station and airport
    if 'düsseldorf' in name_lower and ('hauptbahnhof' in name_lower or 'hbf' in name_lower):
        return 'Düsseldorf Hbf'
    if 'düsseldorf' in name_lower and ('flughafen' in name_lower or 'airport' in name_lower):
        return 'Düsseldorf Flughafen'
    
    # Other major cities - normalize to Hbf form (more aggressive matching)
    city_normalizations = {
        'hannover': 'Hannover Hbf',
        'nürnberg': 'Nürnberg Hbf', 'nuremberg': 'Nürnberg Hbf',
        'leipzig': 'Leipzig Hbf',
        'dresden': 'Dresden Hbf',
        'saarbrücken': 'Saarbrücken Hbf',
        'aachen': 'Aachen Hbf',
        'passau': 'Passau Hbf',
        'freiburg': 'Freiburg(Breisgau) Hbf',
    }
    
    # Match cities that are in our important list (with or without Hbf)
    for city_key, canonical in city_normalizations.items():
        # Check if city name appears in station name
        if city_key in name_lower:
            # If it already has Hbf/Hauptbahnhof, normalize to canonical
            if 'hauptbahnhof' in name_lower or 'hbf' in name_lower:
                return canonical
            # If it's just the city name (no Hbf), check if it matches our important stations
            # For now, only normalize if it's a border city we care about
            if city_key in ['saarbrücken', 'aachen', 'passau', 'freiburg']:
                return canonical
    
    # Border crossings - normalize variants
    if 'lindau' in name_lower and 'reutin' in name_lower:
        return 'Lindau-Reutin'
    if 'lindau' in name_lower and 'hbf' in name_lower:
        return 'Lindau-Reutin'  # Normalize Lindau Hbf to Lindau-Reutin
    if 'rosenheim' in name_lower:
        return 'Rosenheim'
    if 'freilassing' in name_lower:
        return 'Freilassing'
    # Basel variants - normalize to Basel Bad Bf
    if 'basel' in name_lower and ('bad' in name_lower or 'bf' in name_lower):
        return 'Basel Bad Bf'
    if 'basel' in name_lower and 'sbb' in name_lower:
        return 'Basel Bad Bf'  # Basel SBB -> Basel Bad Bf
    
    # Return original if no normalization found
    return station_name

# Define ONLY the most essential stations (absolute top-tier hubs and critical borders)
GERMANY_IMPORTANT_STATIONS = {
    # Top-tier major hubs (the absolute most important)
    'Frankfurt (Main) Hbf',
    'Berlin Hbf',
    'Berlin Südkreuz',  # Major hub
    'München Hbf',
    'Hamburg Hbf',
    'Köln Hbf',
    'Stuttgart Hbf',
    'Düsseldorf Hbf',
    'Hannover Hbf',
    'Nürnberg Hbf',
    'Leipzig Hbf',
    'Dresden Hbf',
    # Critical border crossings and high-traffic international connections only
    'Basel Bad Bf',  # Switzerland/France border
    'Saarbrücken Hbf',  # France border
    'Aachen Hbf',  # Belgium/Netherlands border
    'Passau Hbf',  # Austria border
    'Lindau-Reutin',  # Austria border
    'Rosenheim',  # High-traffic junction near Austria
    'Freilassing',  # Austria border
    'Freiburg(Breisgau) Hbf',  # High-traffic border hub (265 routes)
    # Major airports (only the biggest hubs)
    'Flughafen BER',
    'Düsseldorf Flughafen',
    'München Flughafen',
    'Frankfurt Flughafen',
}

# Normalize Luxembourg station names
def normalize_luxembourg_station(station_name):
    """Normalize Luxembourg station names."""
    if not station_name:
        return station_name
    
    name_lower = station_name.lower()
    if 'luxembourg' in name_lower or 'luxemburg' in name_lower:
        return 'Luxembourg'
    return station_name

# Define important stations for Luxembourg (keep only major ones)
LUXEMBOURG_IMPORTANT_STATIONS = {
    'Luxembourg',
    'Troisvierges',  # Border with Belgium
    'Rodange',  # Border with Belgium
}

def parse_legs(legs_value):
    """Parse legs JSON string into list of stops."""
    if pd.isna(legs_value) or legs_value == '':
        return []
    
    if isinstance(legs_value, str):
        try:
            legs_list = json.loads(legs_value)
        except (json.JSONDecodeError, ValueError):
            return []
    else:
        legs_list = legs_value
    
    return legs_list

def filter_stops_in_legs(legs_list, important_stations):
    """Filter legs to keep only important stations for a country."""
    if not legs_list:
        return []
    
    filtered_legs = []
    for idx, item in enumerate(legs_list):
        if not isinstance(item, list) or len(item) != 2:
            continue
        station_info, times = item
        if not isinstance(station_info, list) or len(station_info) != 2:
            continue
        station_name, country = station_info
        
        # Normalize station name to canonical form
        if country == 'Germany':
            normalized_name = normalize_station_name(station_name)
        elif country == 'Luxembourg':
            normalized_name = normalize_luxembourg_station(station_name)
        else:
            normalized_name = station_name
        
        # Check if this is origin or destination
        is_origin = idx == 0
        is_destination = idx == len(legs_list) - 1
        
        # Keep the stop if it's not in the country we're filtering
        if country not in ['Germany', 'Luxembourg']:
            filtered_legs.append(item)
        # Keep important stations in Germany/Luxembourg
        elif country == 'Germany' and normalized_name in GERMANY_IMPORTANT_STATIONS:
            # Update the station name to normalized form
            filtered_legs.append([[normalized_name, country], times])
        elif country == 'Luxembourg' and normalized_name in LUXEMBOURG_IMPORTANT_STATIONS:
            # Update the station name to normalized form
            filtered_legs.append([[normalized_name, country], times])
        # Only keep origin/destination if they're in other countries (already handled above)
        # OR if they're important stations (already handled above)
        # We do NOT keep non-important origin/destination stops in Germany/Luxembourg
    
    return filtered_legs

# Count station frequency to identify additional hubs (using normalized names)
legs_column = 'legs_filtered' if 'legs_filtered' in routes_df.columns else 'legs_with_countries'
station_frequency = defaultdict(int)

for idx, row in routes_df.iterrows():
    legs_data = row.get(legs_column)
    legs_list = parse_legs(legs_data)
    for item in legs_list:
        if isinstance(item, list) and len(item) == 2:
            station_info = item[0]
            if isinstance(station_info, list) and len(station_info) == 2:
                station_name, country = station_info
                if country == 'Germany':
                    normalized_name = normalize_station_name(station_name)
                    station_frequency[(normalized_name, country)] += 1
                elif country == 'Luxembourg':
                    normalized_name = normalize_luxembourg_station(station_name)
                    station_frequency[(normalized_name, country)] += 1

# Add high-frequency stations to important lists (stations that appear in many routes)
# This helps identify hubs that might not be in our predefined list
# Using an extremely high threshold to be very selective (only major hubs)
HIGH_FREQUENCY_THRESHOLD_GERMANY = 200  # Appears in at least 200 routes for Germany (very selective)
HIGH_FREQUENCY_THRESHOLD_LUXEMBOURG = 20  # Lower threshold for Luxembourg (smaller country)

print(f"Germany predefined stations: {len(GERMANY_IMPORTANT_STATIONS)}")
print(f"Luxembourg predefined stations: {len(LUXEMBOURG_IMPORTANT_STATIONS)}")

# Check which predefined stations actually exist in the data (before adding high-frequency)
germany_in_data = set()
luxembourg_in_data = set()
for (station_name, country), count in station_frequency.items():
    if country == 'Germany' and station_name in GERMANY_IMPORTANT_STATIONS:
        germany_in_data.add(station_name)
    elif country == 'Luxembourg' and station_name in LUXEMBOURG_IMPORTANT_STATIONS:
        luxembourg_in_data.add(station_name)

print(f"\nPredefined stations found in data:")
print(f"  Germany: {len(germany_in_data)}/{len(GERMANY_IMPORTANT_STATIONS)}")
if len(GERMANY_IMPORTANT_STATIONS) - len(germany_in_data) > 0:
    missing = GERMANY_IMPORTANT_STATIONS - germany_in_data
    print(f"    Missing from data: {sorted(missing)}")
print(f"  Luxembourg: {len(luxembourg_in_data)}/{len(LUXEMBOURG_IMPORTANT_STATIONS)}")
if len(LUXEMBOURG_IMPORTANT_STATIONS) - len(luxembourg_in_data) > 0:
    missing = LUXEMBOURG_IMPORTANT_STATIONS - luxembourg_in_data
    print(f"    Missing from data: {sorted(missing)}")

# Add high-frequency stations
high_freq_added_germany = []
high_freq_added_luxembourg = []
for (station_name, country), count in station_frequency.items():
    if country == 'Germany' and count >= HIGH_FREQUENCY_THRESHOLD_GERMANY:
        if station_name not in GERMANY_IMPORTANT_STATIONS:
            GERMANY_IMPORTANT_STATIONS.add(station_name)
            high_freq_added_germany.append((station_name, count))
    elif country == 'Luxembourg' and count >= HIGH_FREQUENCY_THRESHOLD_LUXEMBOURG:
        if station_name not in LUXEMBOURG_IMPORTANT_STATIONS:
            LUXEMBOURG_IMPORTANT_STATIONS.add(station_name)
            high_freq_added_luxembourg.append((station_name, count))

print(f"\nAfter adding high-frequency stations:")
print(f"  Germany important stations: {len(GERMANY_IMPORTANT_STATIONS)}")
print(f"  Luxembourg important stations: {len(LUXEMBOURG_IMPORTANT_STATIONS)}")

if high_freq_added_germany:
    print(f"\nHigh-frequency stations added to Germany (>{HIGH_FREQUENCY_THRESHOLD_GERMANY} routes):")
    for station_name, count in sorted(high_freq_added_germany, key=lambda x: x[1], reverse=True):
        print(f"    {station_name}: {count} routes")
if high_freq_added_luxembourg:
    print(f"\nHigh-frequency stations added to Luxembourg (>{HIGH_FREQUENCY_THRESHOLD_LUXEMBOURG} routes):")
    for station_name, count in sorted(high_freq_added_luxembourg, key=lambda x: x[1], reverse=True):
        print(f"    {station_name}: {count} routes")

# Apply filtering to routes_df
print("\nFiltering routes...")
routes_filtered = routes_df.copy()

for idx, row in routes_df.iterrows():
    legs_data = row.get(legs_column)
    legs_list = parse_legs(legs_data)
    
    if legs_list:
        filtered_legs = filter_stops_in_legs(legs_list, GERMANY_IMPORTANT_STATIONS | LUXEMBOURG_IMPORTANT_STATIONS)
        
        # Update the legs_filtered column
        if legs_column == 'legs_filtered':
            routes_filtered.at[idx, 'legs_filtered'] = json.dumps(filtered_legs) if filtered_legs else None
        else:
            # Create legs_filtered column if it doesn't exist
            if 'legs_filtered' not in routes_filtered.columns:
                routes_filtered['legs_filtered'] = None
            routes_filtered.at[idx, 'legs_filtered'] = json.dumps(filtered_legs) if filtered_legs else None

# Update routes_df
routes_df = routes_filtered

# Verify filtering results (using normalized names for consistency)
print("\nVerifying filtering results...")
stops_by_country_after = defaultdict(lambda: defaultdict(int))

for idx, row in routes_df.iterrows():
    legs_data = row.get('legs_filtered') or row.get(legs_column)
    legs_list = parse_legs(legs_data)
    for item in legs_list:
        if isinstance(item, list) and len(item) == 2:
            station_info = item[0]
            if isinstance(station_info, list) and len(station_info) == 2:
                station_name, country = station_info
                if country == 'Germany':
                    # Normalize to match how we filtered
                    normalized_name = normalize_station_name(station_name)
                    stops_by_country_after[country][normalized_name] += 1
                elif country == 'Luxembourg':
                    # Normalize to match how we filtered
                    normalized_name = normalize_luxembourg_station(station_name)
                    stops_by_country_after[country][normalized_name] += 1

print("\nAfter filtering:")
for country in ['Germany', 'Luxembourg']:
    unique_stations = len(stops_by_country_after[country])
    total_stops = sum(stops_by_country_after[country].values())
    print(f"  {country}: {unique_stations} unique stations, {total_stops} total stops")
    
    # Show which predefined stations actually appear vs which don't
    if country == 'Germany':
        predefined_set = GERMANY_IMPORTANT_STATIONS
    else:
        predefined_set = LUXEMBOURG_IMPORTANT_STATIONS
    
    stations_found = set(stops_by_country_after[country].keys())
    stations_not_found = predefined_set - stations_found
    stations_extra = stations_found - predefined_set
    
    print(f"    Predefined stations found: {len(stations_found & predefined_set)}/{len(predefined_set)}")
    if stations_not_found:
        print(f"    Predefined stations NOT found in data: {sorted(stations_not_found)}")
    if stations_extra:
        print(f"    Extra stations (from high-frequency): {sorted(stations_extra)}")
    
    if unique_stations <= 30:  # Show all if not too many
        print(f"    All stations: {sorted(stops_by_country_after[country].keys())}")

print("\nFiltering complete!")

Germany predefined stations: 24
Luxembourg predefined stations: 3

Predefined stations found in data:
  Germany: 20/24
    Missing from data: ['Basel Bad Bf', 'Düsseldorf Flughafen', 'Flughafen BER', 'München Flughafen']
  Luxembourg: 3/3

After adding high-frequency stations:
  Germany important stations: 24
  Luxembourg important stations: 3

Filtering routes...

Verifying filtering results...

After filtering:
  Germany: 20 unique stations, 2824 total stops
    Predefined stations found: 20/24
    Predefined stations NOT found in data: ['Basel Bad Bf', 'Düsseldorf Flughafen', 'Flughafen BER', 'München Flughafen']
    All stations: ['Aachen Hbf', 'Berlin Hbf', 'Berlin Südkreuz', 'Dresden Hbf', 'Düsseldorf Hbf', 'Frankfurt (Main) Hbf', 'Frankfurt Flughafen', 'Freiburg(Breisgau) Hbf', 'Freilassing', 'Hamburg Hbf', 'Hannover Hbf', 'Köln Hbf', 'Leipzig Hbf', 'Lindau-Reutin', 'München Hbf', 'Nürnberg Hbf', 'Passau Hbf', 'Rosenheim', 'Saarbrücken Hbf', 'Stuttgart Hbf']
  Luxembourg: 3 uniq

### SORT STOPS AND FILL NONE TIMES

In [6]:
import json
import numpy as np
from datetime import datetime, timedelta

def time_to_minutes(time_str):
    """Convert time string (HH:MM) to minutes since midnight."""
    if pd.isna(time_str) or time_str is None or time_str == '':
        return None
    try:
        hours, minutes = map(int, time_str.split(':'))
        return hours * 60 + minutes
    except (ValueError, AttributeError):
        return None

def minutes_to_time(minutes):
    """Convert minutes since midnight to time string (HH:MM)."""
    if minutes is None:
        return None
    hours = minutes // 60
    mins = minutes % 60
    return f"{hours:02d}:{mins:02d}"

def get_sort_key_for_leg(leg, departure_time_minutes):
    """Get a sort key for a leg based on its times, handling midnight rollover."""
    if not isinstance(leg, list) or len(leg) != 2:
        return (float('inf'), float('inf'))
    
    station_info, times = leg
    if not isinstance(times, list) or len(times) != 2:
        return (float('inf'), float('inf'))
    
    arrival_time, departure_time = times
    
    # Convert times to minutes
    arrival_minutes = time_to_minutes(arrival_time)
    departure_minutes = time_to_minutes(departure_time)
    
    # Use arrival time if available, otherwise departure time
    primary_time = arrival_minutes if arrival_minutes is not None else departure_minutes
    
    if primary_time is None:
        return (float('inf'), float('inf'))
    
    # Handle midnight rollover: 
    # - Times >= departure_time are on the same day (day 0)
    # - Times < departure_time are on the next day (day 1), add 1440 minutes
    if departure_time_minutes is not None:
        if primary_time < departure_time_minutes:
            primary_time += 1440  # Add 24 hours for next day
    
    # Secondary key is departure time if available
    secondary_time = departure_minutes if departure_minutes is not None else arrival_minutes
    if secondary_time is not None and departure_time_minutes is not None:
        if secondary_time < departure_time_minutes:
            secondary_time += 1440
    
    return (primary_time, secondary_time if secondary_time is not None else primary_time)

def fill_none_times(legs_sorted, route_departure_time, route_arrival_time):
    """Fill in None times by copying the non-None value from the same stop.
    
    If arrival_time is None but departure_time has a value, set arrival_time = departure_time.
    If departure_time is None but arrival_time has a value, set departure_time = arrival_time.
    """
    if not legs_sorted:
        return legs_sorted
    
    filled_legs = []
    
    for leg in legs_sorted:
        if not isinstance(leg, list) or len(leg) != 2:
            filled_legs.append(leg)
            continue
        
        station_info, times = leg
        if not isinstance(times, list) or len(times) != 2:
            filled_legs.append(leg)
            continue
        
        arrival_time, departure_time = times
        
        # Simple filling: copy the non-None value to the None value
        if arrival_time is None and departure_time is not None:
            arrival_time = departure_time
        elif departure_time is None and arrival_time is not None:
            departure_time = arrival_time
        # If both are None, leave them as None
        
        filled_legs.append([station_info, [arrival_time, departure_time]])
    
    return filled_legs

def sort_and_fill_legs(row):
    """Sort legs chronologically and fill None times for a single route."""
    # Access legs_data - handle both Series and dict-like access
    try:
        legs_data = row['legs_filtered']
    except (KeyError, IndexError, TypeError):
        legs_data = row.get('legs_filtered', None)
    
    # If legs_data is None, return None (preserve None values)
    if legs_data is None:
        return None
    
    # Store original for fallback
    original_legs_data = legs_data
    
    # Parse legs_data - handle both list and JSON string formats
    if isinstance(legs_data, list):
        legs_list = legs_data
    elif isinstance(legs_data, str):
        # Try to parse as JSON
        legs_list = parse_legs(legs_data)
        # If parsing failed (returned empty list), but original string is not empty, use original
        if not legs_list and legs_data.strip() and legs_data.strip() != 'null':
            # Parsing failed but we have data - try to preserve it
            return legs_data
    else:
        # Unknown format, try to convert to JSON
        try:
            legs_list = json.loads(str(legs_data)) if legs_data else []
        except:
            legs_list = []
    
    # CRITICAL: If we have no legs after parsing, preserve original data
    if not legs_list:
        if isinstance(original_legs_data, str) and original_legs_data.strip():
            return original_legs_data
        elif isinstance(original_legs_data, list):
            return json.dumps(original_legs_data)
        else:
            return None
    
    # Get route departure and arrival times
    try:
        route_departure_time = row['departure_time']
        route_arrival_time = row['arrival_time']
    except (KeyError, IndexError, TypeError):
        route_departure_time = row.get('departure_time')
        route_arrival_time = row.get('arrival_time')
    
    # Convert to scalar if needed
    if isinstance(route_departure_time, pd.Series):
        route_departure_time = route_departure_time.iloc[0] if len(route_departure_time) > 0 else None
    if isinstance(route_arrival_time, pd.Series):
        route_arrival_time = route_arrival_time.iloc[0] if len(route_arrival_time) > 0 else None
    
    route_dep_minutes = time_to_minutes(route_departure_time)
    
    # Sort legs based on their times - preserve ALL legs, even those without times
    try:
        # Sort, but ensure all legs are included (those with inf keys go to end)
        legs_sorted = sorted(legs_list, key=lambda leg: get_sort_key_for_leg(leg, route_dep_minutes))
        # Verify we didn't lose any legs
        if len(legs_sorted) != len(legs_list):
            print(f"Warning: Lost {len(legs_list) - len(legs_sorted)} legs during sorting, using original order")
            legs_sorted = legs_list
    except Exception as e:
        # If sorting fails, preserve original order
        print(f"Warning: Sorting failed, preserving original order. Error: {e}")
        legs_sorted = legs_list
    
    # Fill None times - this should preserve all legs
    try:
        legs_filled = fill_none_times(legs_sorted, route_departure_time, route_arrival_time)
        # CRITICAL: Verify we didn't lose any legs during filling
        if len(legs_filled) != len(legs_sorted):
            print(f"Warning: Lost {len(legs_sorted) - len(legs_filled)} legs during time filling, using sorted legs")
            legs_filled = legs_sorted
    except Exception as e:
        # If filling fails, return sorted legs without filling
        print(f"Warning: Time filling failed, using sorted legs without filling. Error: {e}")
        legs_filled = legs_sorted
    
    # FINAL SAFETY CHECK: If we somehow lost all legs, return original data
    if not legs_filled:
        if isinstance(original_legs_data, str):
            return original_legs_data
        elif isinstance(original_legs_data, list):
            return json.dumps(original_legs_data)
        else:
            return json.dumps(legs_list) if legs_list else None
    
    # Return as JSON string
    return json.dumps(legs_filled)

# RESTORE DATA IF NEEDED: Check if legs_filtered is empty and restore from legs_with_countries
print("Checking data integrity...")
empty_count = 0
restored_count = 0

for idx, row in routes_df.iterrows():
    legs_filtered = row.get('legs_filtered')
    legs_with_countries = row.get('legs_with_countries')
    
    # Check if legs_filtered is empty/None but legs_with_countries has data
    legs_filtered_empty = False
    if legs_filtered is None or (isinstance(legs_filtered, str) and legs_filtered.strip() in ['', 'null', '[]']):
        legs_filtered_empty = True
    elif isinstance(legs_filtered, list) and len(legs_filtered) == 0:
        legs_filtered_empty = True
    
    if legs_filtered_empty and legs_with_countries is not None:
        # Try to parse legs_with_countries
        if isinstance(legs_with_countries, str) and legs_with_countries.strip():
            try:
                parsed = json.loads(legs_with_countries)
                if parsed:  # If we have data, restore it
                    routes_df.at[idx, 'legs_filtered'] = legs_with_countries
                    restored_count += 1
                    empty_count += 1
            except:
                pass
        elif isinstance(legs_with_countries, list) and len(legs_with_countries) > 0:
            routes_df.at[idx, 'legs_filtered'] = json.dumps(legs_with_countries)
            restored_count += 1
            empty_count += 1
    elif legs_filtered_empty:
        empty_count += 1

if restored_count > 0:
    print(f"✓ Restored {restored_count} routes from legs_with_countries")
if empty_count > restored_count:
    print(f"⚠ Warning: {empty_count - restored_count} routes still have empty legs_filtered")

# Apply sorting and time filling to all routes
print("\nSorting stops and filling None times...")
routes_sorted = routes_df.copy()

processed_count = 0
error_count = 0
preserved_count = 0

for idx, row in routes_df.iterrows():
    try:
        # Get original data before processing
        original_legs = row.get('legs_filtered')
        
        # Skip if original is empty
        if original_legs is None or (isinstance(original_legs, str) and original_legs.strip() in ['', 'null', '[]']):
            routes_sorted.at[idx, 'legs_filtered'] = original_legs
            continue
        
        sorted_legs = sort_and_fill_legs(row)
        
        # CRITICAL SAFETY CHECK: if sorted_legs is None/empty but original wasn't, preserve original
        if (sorted_legs is None or sorted_legs == '' or sorted_legs == '[]') and original_legs is not None:
            if isinstance(original_legs, str) and original_legs.strip() not in ['', 'null', '[]']:
                # Original was a non-empty string, preserve it
                sorted_legs = original_legs
                preserved_count += 1
            elif isinstance(original_legs, list) and len(original_legs) > 0:
                # Original was a non-empty list, preserve it
                sorted_legs = json.dumps(original_legs)
                preserved_count += 1
        
        routes_sorted.at[idx, 'legs_filtered'] = sorted_legs
        processed_count += 1
    except Exception as e:
        error_count += 1
        # On error, preserve original data
        original_legs = row.get('legs_filtered')
        if original_legs is not None:
            routes_sorted.at[idx, 'legs_filtered'] = original_legs
        print(f"Error processing row {idx}: {e}")

routes_df = routes_sorted

print(f"\nProcessed {processed_count} routes")
if preserved_count > 0:
    print(f"Preserved {preserved_count} routes that would have been lost")
if error_count > 0:
    print(f"Errors encountered: {error_count}")

print("Sorting and time filling complete!")

# Validation: Check that all rows are sorted correctly
print("\nValidating sorting...")
unsorted_count = 0
unsorted_examples = []

for idx, row in routes_df.iterrows():
    legs_data = row.get('legs_filtered')
    
    # Convert to scalar if needed
    if isinstance(legs_data, pd.Series):
        if len(legs_data) == 0:
            continue
        legs_data = legs_data.iloc[0]
    
    # Check for NaN or empty
    try:
        is_na = pd.isna(legs_data)
        if isinstance(is_na, (pd.Series, np.ndarray)):
            is_na = is_na.any() if hasattr(is_na, 'any') else bool(is_na)
        if is_na or (isinstance(legs_data, str) and legs_data.strip() == ''):
            continue
    except (ValueError, TypeError):
        if legs_data is None:
            continue
    
    legs_list = parse_legs(legs_data)
    if not legs_list or len(legs_list) < 2:
        continue
    
    # Get route departure time
    route_departure_time = row.get('departure_time')
    if isinstance(route_departure_time, pd.Series):
        route_departure_time = route_departure_time.iloc[0] if len(route_departure_time) > 0 else None
    route_dep_minutes = time_to_minutes(route_departure_time)
    
    # Check if legs are sorted by comparing consecutive stops
    prev_time = None
    is_sorted = True
    
    for leg in legs_list:
        if not isinstance(leg, list) or len(leg) != 2:
            continue
        
        station_info, times = leg
        if not isinstance(times, list) or len(times) != 2:
            continue
        
        arrival_time, departure_time = times
        arrival_minutes = time_to_minutes(arrival_time)
        
        if arrival_minutes is None:
            # Use departure time if arrival is None
            arrival_minutes = time_to_minutes(departure_time)
        
        if arrival_minutes is None:
            continue
        
        # Handle midnight rollover
        if route_dep_minutes is not None:
            if arrival_minutes < route_dep_minutes:
                arrival_minutes += 1440
        
        if prev_time is not None and arrival_minutes < prev_time:
            is_sorted = False
            break
        
        prev_time = arrival_minutes
    
    if not is_sorted:
        unsorted_count += 1
        if len(unsorted_examples) < 5:
            unsorted_examples.append({
                'index': idx,
                'origin': row.get('route_origin'),
                'destination': row.get('route_destination'),
                'departure': route_departure_time
            })

if unsorted_count == 0:
    print(f"✓ All {len(routes_df)} routes are correctly sorted!")
else:
    print(f"⚠ Warning: {unsorted_count} routes are not correctly sorted")
    if unsorted_examples:
        print("\nExamples of unsorted routes:")
        for ex in unsorted_examples[:5]:
            print(f"  Index {ex['index']}: {ex['origin']} -> {ex['destination']} (dep: {ex['departure']})")

print(f"\nSample route after sorting (index 3095):")
sample_row = routes_df.iloc[3095]
print(f"  Origin: {sample_row['route_origin']}")
print(f"  Destination: {sample_row['route_destination']}")
print(f"  Departure: {sample_row['departure_time']}")
print(f"  Arrival: {sample_row['arrival_time']}")
sample_legs = parse_legs(sample_row['legs_filtered'])
print(f"  Legs ({len(sample_legs)} stops):")
for i, leg in enumerate(sample_legs[:5]):  # Show first 5
    if isinstance(leg, list) and len(leg) == 2:
        station_info, times = leg
        if isinstance(station_info, list) and len(station_info) == 2:
            station_name, country = station_info
            print(f"    {i+1}. {station_name} ({country}): arr={times[0]}, dep={times[1]}")
if len(sample_legs) > 5:
    print(f"    ... ({len(sample_legs) - 5} more stops)")

Checking data integrity...

Sorting stops and filling None times...

Processed 3096 routes
Sorting and time filling complete!

Validating sorting...
✓ All 3096 routes are correctly sorted!

Sample route after sorting (index 3095):
  Origin: Budapest
  Destination: Warsaw
  Departure: 21:25
  Arrival: 14:00
  Legs (10 stops):
    1. Györ (Hungary): arr=23:19, dep=23:19
    2. Lébény-Mosonszentmiklos (Hungary): arr=23:59, dep=00:00
    3. Kimle-Karolyhaza (Hungary): arr=00:05, dep=00:06
    4. Nickelsdorf (Austria): arr=04:17, dep=04:18
    5. Wien Hbf (Austria): arr=05:47, dep=05:47
    ... (5 more stops)


In [7]:
import json

print("\nGrouping legs_filtered by country while preserving per-country order...")

def _lf_country_from_leg(leg):
    """Extract country name from a leg [[name, country], [arr, dep]]."""
    if (isinstance(leg, list) and len(leg) == 2 and
        isinstance(leg[0], list) and len(leg[0]) == 2):
        return leg[0][1]
    return None

def group_legs_filtered_by_country(row):
    """Group legs_filtered by country for a single route.

    - Uses the already time-sorted legs_filtered as input.
    - Groups legs so that all stops from the same country are consecutive.
    - Keeps per-country internal order as produced by the sorter.
    - Ensures the destination country is last, and, when possible,
      the destination stop (matching arrival_time) is the last leg
      within that country.
    """
    legs_data = row.get('legs_filtered')
    if legs_data is None:
        return legs_data

    # Parse to list
    if isinstance(legs_data, str):
        legs_list = parse_legs(legs_data)
    elif isinstance(legs_data, list):
        legs_list = legs_data
    else:
        try:
            legs_list = json.loads(str(legs_data)) if legs_data else []
        except Exception:
            legs_list = []

    if not legs_list or len(legs_list) < 2:
        return legs_data

    # Build full country sequence and first-appearance order
    country_order = []
    countries_seq = []
    for leg in legs_list:
        c = _lf_country_from_leg(leg)
        countries_seq.append(c)
        if c and c not in country_order:
            country_order.append(c)

    # Identify origin and destination countries
    origin_country = next((c for c in countries_seq if c), None)
    dest_country = next((c for c in reversed(countries_seq) if c), None)

    # Ensure destination country is last in the order (if different from origin)
    if dest_country and dest_country != origin_country and dest_country in country_order:
        country_order = [c for c in country_order if c != dest_country] + [dest_country]

    # Group legs by country, preserving internal order
    by_country = {}
    for leg in legs_list:
        c = _lf_country_from_leg(leg) or '__UNKNOWN__'
        by_country.setdefault(c, []).append(leg)

    # Rebuild list in country_order, with destination stop last inside dest country when identifiable
    arrival_time_route = row.get('arrival_time')
    reordered = []
    for c in country_order:
        if c not in by_country:
            continue
        group = by_country[c]
        if c == dest_country and arrival_time_route:
            # Try to move the true destination stop (matching arrival_time) to the end of this group
            dest_idx = None
            for i, leg in enumerate(group):
                if not isinstance(leg, list) or len(leg) != 2:
                    continue
                _, times = leg
                if not isinstance(times, list) or len(times) != 2:
                    continue
                arr_t, dep_t = times
                if arr_t == arrival_time_route or dep_t == arrival_time_route:
                    dest_idx = i
                    break
            if dest_idx is not None and dest_idx != len(group) - 1:
                dest_leg = group.pop(dest_idx)
                group.append(dest_leg)
        reordered.extend(group)

    # Append any leftover (including unknown) countries
    for c, group in by_country.items():
        if c not in country_order:
            reordered.extend(group)

    # Safety: if something went wrong, keep original
    if len(reordered) != len(legs_list):
        return legs_data

    # Preserve original representation type (string vs list)
    if isinstance(legs_data, str):
        return json.dumps(reordered)
    return reordered

regrouped_count = 0
for idx, row in routes_df.iterrows():
    current = row.get('legs_filtered')
    new_val = group_legs_filtered_by_country(row)
    if new_val is not None and new_val != current:
        routes_df.at[idx, 'legs_filtered'] = new_val
        regrouped_count += 1

print(f"Grouped {regrouped_count} routes by country.")


Grouping legs_filtered by country while preserving per-country order...
Grouped 97 routes by country.


### CONVERT TIMES TO ABSOLUTE MINUTES

In [8]:
import json
from datetime import datetime

def parse_datetime_str(dt_str):
    """Parse datetime string like '2026-01-23t07:00:00.000' to datetime object."""
    if pd.isna(dt_str) or dt_str is None or dt_str == '':
        return None
    try:
        # Handle format: 2026-01-23t07:00:00.000
        dt_str = str(dt_str).replace('t', 'T')  # Normalize separator
        # Try parsing with microseconds
        try:
            return datetime.strptime(dt_str, '%Y-%m-%dT%H:%M:%S.%f')
        except ValueError:
            # Try without microseconds
            try:
                return datetime.strptime(dt_str, '%Y-%m-%dT%H:%M:%S')
            except ValueError:
                # Try with just date and time separated by space
                return datetime.strptime(dt_str.replace('T', ' '), '%Y-%m-%d %H:%M:%S')
    except (ValueError, AttributeError) as e:
        return None

def time_to_minutes_absolute(time_str):
    """Convert time string (HH:MM) to minutes since midnight."""
    if pd.isna(time_str) or time_str is None or time_str == '':
        return None
    try:
        hours, minutes = map(int, str(time_str).split(':'))
        return hours * 60 + minutes
    except (ValueError, AttributeError):
        return None

# ===== PROCESS FLIGHTS_DF =====
print("Processing flights_df...")

def convert_flight_times_to_absolute(row):
    """Convert flight departure and arrival times to absolute minutes."""
    departure_scheduled = row.get('departure_scheduled')
    arrival_scheduled = row.get('arrival_scheduled')
    
    if pd.isna(departure_scheduled) or departure_scheduled is None:
        return None, None
    
    # Parse departure datetime
    dep_dt = parse_datetime_str(departure_scheduled)
    if dep_dt is None:
        return None, None
    
    # Day 1 starts at 00:00 of the departure date
    # Calculate absolute minutes for departure (minutes since 00:00 of departure date)
    dep_absolute = dep_dt.hour * 60 + dep_dt.minute
    
    # Parse arrival datetime
    arr_dt = parse_datetime_str(arrival_scheduled) if not pd.isna(arrival_scheduled) and arrival_scheduled is not None else None
    
    if arr_dt is None:
        return dep_absolute, None
    
    # Calculate absolute minutes for arrival
    # If arrival is on the same day as departure
    if arr_dt.date() == dep_dt.date():
        arr_absolute = arr_dt.hour * 60 + arr_dt.minute
    else:
        # Arrival is on a different day - calculate days difference
        days_diff = (arr_dt.date() - dep_dt.date()).days
        arr_absolute = days_diff * 1440 + arr_dt.hour * 60 + arr_dt.minute
    
    return dep_absolute, arr_absolute

# Apply conversion to flights_df
flights_absolute = []
for idx, row in flights_df.iterrows():
    dep_abs, arr_abs = convert_flight_times_to_absolute(row)
    flights_absolute.append({
        'departure_scheduled_absolute': dep_abs,
        'arrival_scheduled_absolute': arr_abs
    })

# Add new columns to flights_df
flights_df['departure_scheduled_absolute'] = [x['departure_scheduled_absolute'] for x in flights_absolute]
flights_df['arrival_scheduled_absolute'] = [x['arrival_scheduled_absolute'] for x in flights_absolute]

print(f"✓ Processed {len(flights_df)} flights")
print(f"  Departure times converted: {flights_df['departure_scheduled_absolute'].notna().sum()}")
print(f"  Arrival times converted: {flights_df['arrival_scheduled_absolute'].notna().sum()}")

# ===== PROCESS ROUTES_DF =====
print("\nProcessing routes_df...")

def convert_route_times_to_absolute(row):
    """Convert route leg times to absolute minutes based on first stop."""
    legs_data = row.get('legs_filtered')
    
    if pd.isna(legs_data) or legs_data is None or legs_data == '':
        return None
    
    # Parse legs
    legs_list = parse_legs(legs_data)
    if not legs_list:
        return None
    
    # First stop is the reference (day 1, minute 0)
    # Get the time of the first stop
    if not isinstance(legs_list[0], list) or len(legs_list[0]) != 2:
        return None
    
    first_station_info, first_times = legs_list[0]
    if not isinstance(first_times, list) or len(first_times) != 2:
        return None
    
    # Use departure time of first stop as reference, or arrival if departure is None
    first_time_str = first_times[1] if first_times[1] is not None else first_times[0]
    if first_time_str is None:
        return None
    
    first_time_minutes = time_to_minutes_absolute(first_time_str)
    if first_time_minutes is None:
        return None
    
    # Convert all legs to absolute minutes
    legs_absolute = []
    prev_absolute_time = first_time_minutes  # Track previous time to handle midnight rollover
    
    for leg in legs_list:
        if not isinstance(leg, list) or len(leg) != 2:
            continue
        
        station_info, times = leg
        if not isinstance(times, list) or len(times) != 2:
            continue
        
        arrival_time_str, departure_time_str = times
        
        # Convert arrival time
        arrival_minutes = time_to_minutes_absolute(arrival_time_str)
        if arrival_minutes is not None:
            # Handle midnight rollover: if arrival is earlier than previous time, it's next day
            if arrival_minutes < prev_absolute_time % 1440:
                # This is on the next day - find which day
                current_day = prev_absolute_time // 1440
                arrival_absolute = current_day * 1440 + 1440 + arrival_minutes
            else:
                # Same day as previous
                current_day = prev_absolute_time // 1440
                arrival_absolute = current_day * 1440 + arrival_minutes
        else:
            arrival_absolute = None
        
        # Convert departure time
        departure_minutes = time_to_minutes_absolute(departure_time_str)
        if departure_minutes is not None:
            # Use arrival time as reference if available, otherwise use previous time
            if arrival_absolute is not None:
                # Departure should be >= arrival (at same stop)
                if departure_minutes < arrival_absolute % 1440:
                    # Departure is on next day relative to arrival
                    arrival_day = arrival_absolute // 1440
                    departure_absolute = arrival_day * 1440 + 1440 + departure_minutes
                else:
                    # Same day as arrival
                    arrival_day = arrival_absolute // 1440
                    departure_absolute = arrival_day * 1440 + departure_minutes
            else:
                # No arrival time, use previous time as reference
                if departure_minutes < prev_absolute_time % 1440:
                    current_day = prev_absolute_time // 1440
                    departure_absolute = current_day * 1440 + 1440 + departure_minutes
                else:
                    current_day = prev_absolute_time // 1440
                    departure_absolute = current_day * 1440 + departure_minutes
        else:
            departure_absolute = None
        
        # Update previous time for next iteration
        if departure_absolute is not None:
            prev_absolute_time = departure_absolute
        elif arrival_absolute is not None:
            prev_absolute_time = arrival_absolute
        
        # Store absolute times
        legs_absolute.append([station_info, [arrival_absolute, departure_absolute]])
    
    return json.dumps(legs_absolute) if legs_absolute else None

# Apply conversion to routes_df
routes_absolute = []
processed_routes = 0
error_routes = 0

for idx, row in routes_df.iterrows():
    try:
        legs_absolute_json = convert_route_times_to_absolute(row)
        routes_absolute.append(legs_absolute_json)
        if legs_absolute_json is not None:
            processed_routes += 1
    except Exception as e:
        routes_absolute.append(None)
        error_routes += 1
        if error_routes <= 5:  # Print first 5 errors
            print(f"  Warning: Error processing route {idx}: {e}")

# Add new column to routes_df
routes_df['legs_filtered_absolute'] = routes_absolute
print(f"✓ Processed {len(routes_df)} routes")
print(f"  Routes with absolute times: {processed_routes}")
if error_routes > 0:
    print(f"  Routes with errors: {error_routes}")

# Show sample
print("\nSample conversion:")
sample_idx = 3095 if len(routes_df) > 3095 else 0
sample_row = routes_df.iloc[sample_idx]
print(f"  Route {sample_idx}: {sample_row.get('route_origin', 'N/A')} -> {sample_row.get('route_destination', 'N/A')}")
print(f"  Original legs_filtered: {str(sample_row.get('legs_filtered', ''))[:100]}...")
print(f"  Absolute legs_filtered_absolute: {str(sample_row.get('legs_filtered_absolute', ''))[:100]}...")

print("\n✓ Conversion to absolute minutes complete!")

Processing flights_df...
✓ Processed 91342 flights
  Departure times converted: 91342
  Arrival times converted: 91342

Processing routes_df...
✓ Processed 3096 routes
  Routes with absolute times: 3096

Sample conversion:
  Route 3095: Budapest -> Warsaw
  Original legs_filtered: [[["Gy\u00f6r", "Hungary"], ["23:19", "23:19"]], [["L\u00e9b\u00e9ny-Mosonszentmiklos", "Hungary"], ...
  Absolute legs_filtered_absolute: [[["Gy\u00f6r", "Hungary"], [1399, 1399]], [["L\u00e9b\u00e9ny-Mosonszentmiklos", "Hungary"], [1439,...

✓ Conversion to absolute minutes complete!


### Map all absolute times to UTC+0 to avoid consistency issues

In [9]:
import pandas as pd

# Normalize flight absolute times to UTC+0
print("Converting flights_df absolute times to UTC+0 using corrected offsets...")

# 1. Define Standard Country Groups (Baselines)
# ---------------------------------------------
# These cover the majority of airports in these countries.
# Note: Spain (ES) is UTC+1, Portugal (PT) is UTC+0 by default here.
UTC0_COUNTRIES = {"GB", "IE", "PT", "IS"}  # Added Iceland (IS) to UTC+0 baseline
UTC2_COUNTRIES = {"FI", "EE", "LV", "LT", "RO", "BG", "GR", "CY", "UA", "MD"} # Added Moldova (MD)

# 2. Define Specific Airport Overrides
# ------------------------------------
# This dictionary maps specific IATA codes to their offset in MINUTES.
# It overrides the country-level default.

AIRPORT_OVERRIDES = {
    # --- UTC -1 (Azores) ---
    "PDL": -60, "TER": -60, "FLW": -60, "CVU": -60, "HOR": -60, 
    "PIX": -60, "GRW": -60, "SJZ": -60, "SMA": -60,

    # --- UTC +0 (Islands & Crown Dependencies often misidentified) ---
    # Canary Islands (Spain is UTC+1, but Canaries are UTC+0)
    "LPA": 0, "TFS": 0, "TFN": 0, "ACE": 0, "FUE": 0, "SPC": 0, "VDE": 0, "GMZ": 0,
    # Madeira (Portugal is UTC+0, so this matches baseline, but good to be explicit if logic changes)
    "FNC": 0, "PXO": 0,
    # Faroe Islands
    "FAE": 0,
    # Crown Dependencies (Jersey, Guernsey, Isle of Man, Alderney)
    "JER": 0, "GCI": 0, "IOM": 0, "ACI": 0,
    # Iceland (Explicit overrides just in case Country code is missing/wrong)
    "AEY": 0, "EGS": 0, "GRY": 0, "IFJ": 0, "RKV": 0, "VPN": 0, "KEF": 0, "HFN": 0, "BIU": 0,

    # --- UTC +2 (Moldova Override if country code fails) ---
    "RMO": 120, "KIV": 120, # Chisinau (Old & New codes)

    # --- UTC +3 (Turkey, Western Russia, Belarus, Crimea) ---
    # Turkey (Country is usually UTC+3 year-round now)
    "IST": 180, "SAW": 180, "AYT": 180, "ESB": 180, "ADB": 180, "DLM": 180, 
    "BJV": 180, "TZX": 180, "GZP": 180, "TEQ": 180,
    # Crimea
    "SIP": 180,
    # Belarus
    "MSQ": 180, "MVQ": 180, "BQT": 180, "GME": 180,
    # Western Russia (Moscow Time Zone)
    "LED": 180, "SVO": 180, "DME": 180, "VKO": 180, "ZIA": 180, 
    "AER": 180, "AAQ": 180, "KRR": 180, "ROV": 180, "VOZ": 180,
    "KZN": 180, "GSV": 180, "MCX": 180, "MRV": 180, "OGZ": 180,
    "SCW": 180, "STW": 180, "ARH": 180, "MMK": 180, "NNM": 180,
    "IWA": 180, "KVX": 180, "NAL": 180, "KMW": 180,

    # --- UTC +4 (Samara Time) ---
    "KUF": 240, 
    
    # --- UTC +5 (Yekaterinburg Time) ---
    "UFA": 300, "SVX": 300, "PEE": 300,
    "UKG": 300, # Ust-Kamenogorsk (Kazakhstan)

    # --- UTC +7 (Novosibirsk/Krasnoyarsk Time) ---
    "OVB": 420, "NOZ": 420, "KJA": 420,

    # --- UTC -9 (Alaska Outlier) ---
    "SYS": -540, 
}


# Work only with European airports from airports_df
# Note: Some Russian/Turkish airports might be listed as AS (Asia) in some datasets.
# We ensure we include them if they are in our target list.
europe_airports = airports_df.copy() 

def _get_offset_minutes(row):
    iata = str(row["iata_code"]).upper()
    
    # A. Check Specific Overrides first
    if iata in AIRPORT_OVERRIDES:
        return AIRPORT_OVERRIDES[iata]
    
    # B. Fallback to Country Logic
    iso = str(row["iso_country"]).upper() if pd.notna(row["iso_country"]) else ""
    
    if iso in UTC0_COUNTRIES:
        return 0
    if iso in UTC2_COUNTRIES:
        return 120
    if iso == "TR": # Turkey Fallback
        return 180
    if iso == "RU": # Russia Default Fallback (Western Russia)
        return 180
    if iso == "BY": # Belarus Default Fallback
        return 180
        
    # C. Default for the rest of Central Europe (DE, FR, ES, IT, etc.)
    return 60

# Compute per-airport offset in minutes
europe_airports["utc_offset_minutes"] = europe_airports.apply(_get_offset_minutes, axis=1)

# Build IATA → offset mapping
iata_offset_map = (
    europe_airports
    .dropna(subset=["iata_code"])
    .set_index("iata_code")["utc_offset_minutes"]
    .to_dict()
)

# Helper to get offset
def get_airport_offset(iata_code):
    if not isinstance(iata_code, str):
        return 0
    return iata_offset_map.get(iata_code, 0) # Default to 0 if unknown

# 4. Apply to flights_df

def _adjust_dep(row):
    # Get offset and subtract it to get to UTC
    # e.g., UTC+1 (60) -> 12:00 Local - 60min = 11:00 UTC
    offset = get_airport_offset(row["departure_airport"])
    val = row["departure_scheduled_absolute"]
    return val - offset if pd.notna(val) else val

def _adjust_arr(row):
    offset = get_airport_offset(row["arrival_airport"])
    val = row["arrival_scheduled_absolute"]
    return val - offset if pd.notna(val) else val

flights_df["departure_scheduled_absolute_utc"] = flights_df.apply(_adjust_dep, axis=1)
flights_df["arrival_scheduled_absolute_utc"] = flights_df.apply(_adjust_arr, axis=1)

print("✓ flights_df absolute times converted to UTC+0 with corrected outlier logic.")

Converting flights_df absolute times to UTC+0 using corrected offsets...
✓ flights_df absolute times converted to UTC+0 with corrected outlier logic.


In [10]:
# Drop rows still not consistent, they are just scheduling errors that make no sense and are an insignificant fraction

for (i, row) in flights_df.iterrows():
    if row["departure_scheduled_absolute_utc"] >= row["arrival_scheduled_absolute_utc"]:
        flights_df.drop(i, inplace=True)

In [11]:
import json
from functools import lru_cache

print("Converting routes_df absolute leg times to UTC+0...")

# We will use country names from the legs to infer UTC offsets, similar to flights_df logic.
# Offsets are in MINUTES and represent local_time = UTC + offset → UTC = local_time - offset.

UTC0_COUNTRIES_ISO = {"GB", "IE", "PT", "IS"}
UTC2_COUNTRIES_ISO = {"FI", "EE", "LV", "LT", "RO", "BG", "GR", "CY", "UA", "MD"}

# Map non-standard country labels used in routes (e.g. "England") to ISO alpha-2 codes.
SPECIAL_COUNTRY_NAME_MAP = {
    "england": "GB",
    "scotland": "GB",
    "wales": "GB",
    "northern ireland": "GB",
}

@lru_cache(maxsize=None)
def _country_name_to_iso(country_name: str):
    """Best-effort mapping from leg country label to ISO alpha-2 code."""
    if not isinstance(country_name, str) or not country_name.strip():
        return None
    name_norm = country_name.strip().lower()

    # Handle common special labels first
    if name_norm in SPECIAL_COUNTRY_NAME_MAP:
        return SPECIAL_COUNTRY_NAME_MAP[name_norm]

    # If already looks like an ISO code
    if len(country_name) == 2 and country_name.isalpha():
        return country_name.upper()

    # Fallback to pycountry fuzzy search
    try:
        match = pycountry.countries.search_fuzzy(country_name)[0]
        return match.alpha_2
    except Exception:
        return None

@lru_cache(maxsize=None)
def _get_country_offset_minutes(country_name: str) -> int:
    """Return UTC offset in minutes for a leg country label.

    Default for most of central Europe is +60 (UTC+1), matching the flights logic.
    """
    iso = _country_name_to_iso(country_name) or ""

    if iso in UTC0_COUNTRIES_ISO:
        return 0
    if iso in UTC2_COUNTRIES_ISO:
        return 120
    if iso == "TR":  # Turkey
        return 180
    if iso in {"RU", "BY"}:  # Western Russia / Belarus default
        return 180

    # Default: central Europe (DE, FR, ES, IT, etc.) → UTC+1
    return 60


def convert_legs_absolute_to_utc(legs_absolute_value):
    """Convert a legs_filtered_absolute JSON/list into UTC+0 minutes based on country offsets."""
    if pd.isna(legs_absolute_value) or legs_absolute_value in (None, "", "null", "[]"):
        return None

    legs_list = parse_legs(legs_absolute_value)
    if not legs_list:
        return None

    legs_utc = []

    for leg in legs_list:
        if not isinstance(leg, list) or len(leg) != 2:
            continue
        station_info, times = leg
        if not isinstance(station_info, list) or len(station_info) != 2:
            continue
        if not isinstance(times, list) or len(times) != 2:
            continue

        station_name, country_name = station_info
        arr_abs, dep_abs = times

        offset = _get_country_offset_minutes(country_name)

        arr_utc = arr_abs - offset if arr_abs is not None else None
        dep_utc = dep_abs - offset if dep_abs is not None else None

        legs_utc.append([[station_name, country_name], [arr_utc, dep_utc]])

    return json.dumps(legs_utc) if legs_utc else None

# Apply conversion to routes_df
routes_df["legs_filtered_absolute_utc"] = routes_df["legs_filtered_absolute"].apply(convert_legs_absolute_to_utc)

print("✓ routes_df absolute leg times converted to UTC+0 and stored in 'legs_filtered_absolute_utc'.")

Converting routes_df absolute leg times to UTC+0...
✓ routes_df absolute leg times converted to UTC+0 and stored in 'legs_filtered_absolute_utc'.


### ADD COUNTRY NAME TO AIRPORTS

In [12]:
# Keep only airports that are actually used in flights_df
used_airports = pd.unique(
    pd.concat(
        [flights_df["departure_airport"], flights_df["arrival_airport"]],
        ignore_index=True,
    )
)

airports_df = airports_df[airports_df["iata_code"].isin(used_airports)].copy()

# Add human-readable country name from ISO country code
import pycountry


def iso_country_to_name(iso_code):
    if pd.isna(iso_code):
        return None
    code = str(iso_code).strip()
    if not code:
        return None

    try:
        if len(code) == 2:
            country = pycountry.countries.get(alpha_2=code.upper())
        elif len(code) == 3:
            country = pycountry.countries.get(alpha_3=code.upper())
        else:
            # Fallback: attempt fuzzy search on the label itself
            country = pycountry.countries.search_fuzzy(code)[0]
        return country.name if country else None
    except Exception:
        return None


airports_df["country_name"] = airports_df["iso_country"].apply(iso_country_to_name)

In [13]:
# Manual hard-coded fixes for country_name after general iso → name mapping

# 1) Make United Kingdom consistent with routes_df, which uses "England"
uk_before = (airports_df["country_name"] == "United Kingdom").sum()
airports_df.loc[airports_df["country_name"] == "United Kingdom", "country_name"] = "England"
uk_after = (airports_df["country_name"] == "United Kingdom").sum()

# 2) Simplify Moldova naming
md_before = (airports_df["country_name"] == "Moldova, Republic of").sum()
airports_df.loc[airports_df["country_name"] == "Moldova, Republic of", "country_name"] = "Moldova"
md_after = (airports_df["country_name"] == "Moldova, Republic of").sum()

print(f"Re-labeled 'United Kingdom' → 'England' for {uk_before} rows; remaining UK labels: {uk_after}.")
print(f"Re-labeled 'Moldova, Republic of' → 'Moldova' for {md_before} rows; remaining old Moldova labels: {md_after}.")

Re-labeled 'United Kingdom' → 'England' for 47 rows; remaining UK labels: 0.
Re-labeled 'Moldova, Republic of' → 'Moldova' for 1 rows; remaining old Moldova labels: 0.


### SAVE FINAL DATASETS TO BE USED IN THE GRAPH CREATION

In [14]:
routes_df.to_csv("../data/final_data/routes_df.csv", index=False)
flights_df.to_csv("../data/final_data/flights_df.csv", index=False)
airports_df.to_csv("../data/final_data/airports_df.csv", index=False)
